In [1]:
!pip install pandas


# Importing required Libraries

In [2]:
import pandas as pd
import os


# Creating a data FOlder

In [5]:
data_folder = "data/"
os.makedirs(data_folder, exist_ok=True)


# Reding the datasets into pandas

In [7]:
df = pd.read_csv("./data/2024_flavors_of_cacoa.tsv", sep='\t')


# Step 2: Data Transformation

In [18]:
# Converting string to float

In [22]:
print(df.columns)


Index(['REF', 'Company (Manufacturer)', 'Company Location', 'Review Date',
       'Country of Bean Origin', 'Specific Bean Origin or Bar Name',
       'Cocoa Percent', 'Ingredients', 'Most Memorable Characteristics',
       'Rating'],
      dtype='object')


In [24]:
df["Cocoa Percent"] = df["Cocoa Percent"].str.replace("%", "").astype(float) / 100


In [34]:
#Splitting Ingredients
# Ensure 'Ingredients' column has valid values
df["Ingredients"] = df["Ingredients"].fillna("").str.strip()  # Remove NaN and whitespace

# Extract ingredient count safely
df["ingredient_count"] = df["Ingredients"].str.extract(r"^(\d+)")  # Extract the number at the start
df["ingredient_count"] = pd.to_numeric(df["ingredient_count"], errors="coerce")  # Convert to float, setting errors to NaN

# Fill NaN values with 0 (if necessary)
df["ingredient_count"] = df["ingredient_count"].fillna(0).astype(int)

# Define ingredient codes
ingredient_codes = ["B", "C", "L", "S", "S*", "Sa", "V"]

# Create separate columns for each ingredient safely
for code in ingredient_codes:
    df[code] = df["Ingredients"].apply(lambda x: 1 if code in x else 0)



In [36]:
# Convert characteristics into separate columns
characteristics = df["Most Memorable Characteristics"].str.get_dummies(sep=", ")

# Identify characteristics that appear at least 20 times
common_characteristics = characteristics.sum().sort_values(ascending=False)
common_characteristics = common_characteristics[common_characteristics >= 20].index

# Add these common characteristics as new columns
df = df.join(characteristics[common_characteristics])


In [38]:
df.drop(columns=["Ingredients", "Most Memorable Characteristics"], inplace=True)


In [40]:
df.to_csv("./data/cleaned_data_2025_flavors_of_cacao.csv", index=False)


In [42]:
df_reduced = df[["Review Date", "Country of Bean Origin", "Cocoa Percent", "Rating"]]

# Save the reduced dataset
df_reduced.to_csv("./data/data_reduced_2025_flavors_of_cacao.csv", index=False)
df_reduced.to_json("./data/data_reduced_2025_flavors_of_cacao.json", orient="records")


In [44]:
df_filtered = df[
    (df["Rating"] >= 3.25) &
    (df["Cocoa Percent"].between(0.65, 0.75)) &
    (df["Review Date"].between(2018, 2021)) &
    (df[["fatty", "earthy", "roasty"]].sum(axis=1) > 0)
]


In [46]:
df_filtered.to_csv("./data/data_filtered_2025_flavors_of_cacao.csv", index=False)
df_filtered.to_json("./data/data_filtered_2025_flavors_of_cacao.json", orient="records")
